# 04 — Step 3: Temporal Profiling

**Goal**: Plot CRA across denoising timesteps to identify when reflection attention peaks.

With Flux-schnell's 4-step schedule, we classify timesteps as early (0), mid (1-2), or late (3).

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import matplotlib.pyplot as plt

from scripts.config import EXPERIMENTS_DIR, NUM_INFERENCE_STEPS
from scripts.metrics import compute_temporal_profiles, identify_peak_timestep
from scripts.visualization import plot_temporal_profiles

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP3_DIR = EXPERIMENTS_DIR / "step3_temporal"
STEP3_DIR.mkdir(parents=True, exist_ok=True)

# Load Step 1 results
with open(STEP1_DIR / "step1_results.pkl", "rb") as f:
    step1 = pickle.load(f)

candidates = step1["candidates"]
mirror_data = step1["mirror_attention_data"]
nonmirror_data = step1["nonmirror_attention_data"]

print(f"Loaded {len(mirror_data)} mirror and {len(nonmirror_data)} non-mirror samples")
print(f"Top candidates: {[(b, h) for b, h, _ in candidates[:5]]}")

In [ ]:
# Compute temporal CRA profiles for mirror images
mirror_profiles = compute_temporal_profiles(
    mirror_data, candidates[:5], NUM_INFERENCE_STEPS
)
nonmirror_profiles = compute_temporal_profiles(
    nonmirror_data, candidates[:5], NUM_INFERENCE_STEPS
)

# Display profiles
print("\nMirror CRA Temporal Profiles:")
for (b, h), profile in mirror_profiles.items():
    steps = "  ".join(f"t{t}={v:.4f}" for t, v in enumerate(profile))
    print(f"  B{b}H{h}: {steps}")

print("\nNon-Mirror CRA Temporal Profiles:")
for (b, h), profile in nonmirror_profiles.items():
    steps = "  ".join(f"t{t}={v:.4f}" for t, v in enumerate(profile))
    print(f"  B{b}H{h}: {steps}")

In [ ]:
# Identify peak timestep
peak_t = identify_peak_timestep(mirror_profiles)
print(f"\nPeak timestep for mirror CRA: t={peak_t}")
labels = ["early", "mid-early", "mid-late", "late"]
print(f"Classification: {labels[peak_t]}")

In [ ]:
# Plot mirror temporal profiles
fig = plot_temporal_profiles(
    mirror_profiles,
    title="Temporal CRA Profile — Mirror Images",
)
fig.savefig(STEP3_DIR / "temporal_cra_mirror.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot non-mirror for comparison
fig = plot_temporal_profiles(
    nonmirror_profiles,
    title="Temporal CRA Profile — Non-Mirror Images",
)
fig.savefig(STEP3_DIR / "temporal_cra_nonmirror.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Side-by-side comparison: mirror vs non-mirror temporal profiles
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
timesteps = list(range(NUM_INFERENCE_STEPS))

for (b, h), profile in mirror_profiles.items():
    axes[0].plot(timesteps, profile, marker='o', linewidth=2, label=f'B{b}H{h}')
axes[0].set_title('Mirror Images')
axes[0].set_xlabel('Denoising Step')
axes[0].set_ylabel('CRA')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(timesteps)

for (b, h), profile in nonmirror_profiles.items():
    axes[1].plot(timesteps, profile, marker='o', linewidth=2, label=f'B{b}H{h}')
axes[1].set_title('Non-Mirror Images')
axes[1].set_xlabel('Denoising Step')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(timesteps)

fig.suptitle('Temporal CRA Comparison', fontsize=14)
fig.tight_layout()
fig.savefig(STEP3_DIR / 'temporal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save results
results = {
    "mirror_profiles": mirror_profiles,
    "nonmirror_profiles": nonmirror_profiles,
    "peak_timestep": peak_t,
}
with open(STEP3_DIR / "step3_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP3_DIR}")